# Historical reproduction: MARBERT vs AraBERT (audit item 1)

**Paper:** Abdul-Mageed, Elmadany & Nagoudi (2021), *ARBERT & MARBERT*, ACL-IJCNLP.
**Protocol (as in the paper):** train on the **full** training set, score the test set after every epoch,
report the **best epoch on test** with the published metric. This is deliberately the historical
(best-on-test) protocol and must be labelled as such in the report.

| Dataset | Metric | Published MARBERT / AraBERT |
|---|---|---|
| ArSarcasm-Sent | F1-PN | 71.50 / 63.50 |
| ASTD (trained on CAMelSent) | F1-PN | 78.00 / 72.00 |
| AJGT | Accuracy | 96.11 / 92.22 |

### How to use
1. `Runtime → Change runtime type → GPU` (T4 is enough).
2. Put Member 1's audited data in Google Drive (see **Section 3** for the expected layout).
3. Edit **Section 2 (Config)** if your folder differs, then `Runtime → Run all`.
4. Runs are **resumable**: if Colab disconnects, just run all cells again. Finished runs are skipped,
   interrupted runs continue from the last saved epoch.
5. When all 18 runs are done, **Section 9** builds the published-vs-ours table and the verdicts.

## 1. Environment: GPU check, Google Drive, package versions

In [ ]:
import os, sys, json, subprocess, platform, time
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

import shutil as _sh
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip() if _sh.which("nvidia-smi") else "no NVIDIA GPU found"
if IN_COLAB and gpu_info == "no NVIDIA GPU found":
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
print("GPU:", gpu_info)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4, 580.82.07, 23034 MiB


In [ ]:
# Key packages are locked on the first session and re-installed with the SAME versions in later sessions,
# so all 18 runs use identical software even if Colab updates its defaults in between.
PROJECT_DIR = "/content/drive/MyDrive/marbert_repro" if IN_COLAB else os.path.abspath("marbert_repro")
os.makedirs(PROJECT_DIR, exist_ok=True)
LOCK_FILE = f"{PROJECT_DIR}/env_lock.json"
LOCKED_PKGS = ["transformers", "tokenizers", "huggingface_hub", "scikit-learn", "safetensors"]

from importlib.metadata import version as _v
if os.path.exists(LOCK_FILE):
    lock = json.load(open(LOCK_FILE))
    need = [f"{p}=={lock[p]}" for p in LOCKED_PKGS if p in lock and _v(p) != lock[p]]
    if need:
        print("Installing locked versions:", need)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
        raise SystemExit(">>> Locked versions installed. Do Runtime -> Restart session, then Run all again.")
    import torch
    if torch.__version__ != lock.get("torch"):
        print(f"WARNING: torch {torch.__version__} differs from locked {lock.get('torch')} - note it in departures.md")
else:
    import torch
    lock = {p: _v(p) for p in LOCKED_PKGS}
    lock.update({"torch": torch.__version__, "python": platform.python_version(), "gpu_first_session": gpu_info})
    json.dump(lock, open(LOCK_FILE, "w"), indent=2)
    print("Created", LOCK_FILE)
print(json.dumps(lock, indent=2))

{
  "transformers": "5.22.1",
  "tokenizers": "0.23.3",
  "huggingface_hub": "1.33.0",
  "scikit-learn": "1.6.1",
  "safetensors": "0.8.0",
  "torch": "2.14.1+cu130",
  "python": "3.13.16",
  "gpu_first_session": "Tesla T4, 580.82.07, 15360 MiB"
}


## 2. Config
Everything the paper specifies is here. Change only paths/seeds unless you log a departure.

In [ ]:
DATA_DIR  = f"{PROJECT_DIR}/data"                     # Member 1's audited files
OUT_DIR   = f"{PROJECT_DIR}/outputs/historical"        # predictions + metrics (keep forever)
CKPT_DIR  = f"{PROJECT_DIR}/checkpoints/historical"    # resume checkpoints (deleted when a run finishes)

MODELS = {
    "marbert": "UBC-NLP/MARBERT",                 # original MARBERT, NOT MARBERTv2
    "arabert": "aubmindlab/bert-base-arabertv01", # AraBERT not named in [1]; our fixed choice (logged)
}

DATASETS = {
    # labels: order = class ids. metric: historical metric used to pick the best epoch and compare.
    "arsarcasm": {"labels": ["negative", "neutral", "positive"], "metric": "f1_pn",
                  "published": {"marbert": 71.50, "arabert": 63.50}},
    "astd":      {"labels": ["negative", "neutral", "positive"], "metric": "f1_pn",
                  "published": {"marbert": 78.00, "arabert": 72.00}},
    "ajgt":      {"labels": ["negative", "positive"], "metric": "accuracy",
                  "published": {"marbert": 96.11, "arabert": 92.22}},
}

# Hyperparameters reported in [1]
HP = dict(lr=2e-6, batch_size=32, max_len=256, epochs=25, weight_decay=0.0,
          optimizer="Adam (torch.optim.Adam)", scheduler="constant (unreported in [1]; assumption)", warmup=0)

SEEDS = [42, 43, 44]
RUN_ORDER = ["astd", "arsarcasm", "ajgt"]   # longest first

# Implementation choices NOT specified in [1] - each one is listed in departures.md
USE_AMP = True            # fp16 mixed precision (faster on T4). Set False for full fp32.
PREPROC = dict(replace_user=True, replace_url=True)  # @mentions -> USER, links -> URL (verify vs MARBERT model card)
TEXT_COL, LABEL_COL = "text", "label"
# Resume checkpoints (~2 GB each for MARBERT incl. optimizer; writing one to Drive takes ~20-60 s).
# Save every N epochs per dataset; 0 = never (short runs simply restart if Colab disconnects).
CKPT_EVERY = {"astd": 5, "arsarcasm": 0, "ajgt": 0}

## 3. Data
Expected layout in Drive (CSV, UTF-8, columns `text` and `label`):
```
marbert_repro/data/
  arsarcasm/train.csv   arsarcasm/test.csv
  astd/train.csv        astd/test.csv      # train = merged CAMelSent (train+dev of ArSAS, ASTD, SemEval, ArSenTD-LEV); test = ASTD test
  ajgt/train.csv        ajgt/test.csv      # split agreed with Member 1 (no official split - log it)
```
Labels may be `POS/NEG/NEU`, `positive/negative/neutral`, `Positive/Negative`, etc.; they are normalised below.
Any other label value stops the notebook so it can be fixed in the data audit, not silently dropped.

In [ ]:
import re, random, hashlib
import numpy as np, pandas as pd

LABEL_ALIASES = {"pos": "positive", "positive": "positive", "neg": "negative", "negative": "negative",
                 "neu": "neutral", "neut": "neutral", "neutral": "neutral"}

URL_RE  = re.compile(r"(https?://\S+|www\.\S+)")
USER_RE = re.compile(r"@\w+")
def preprocess(t: str) -> str:
    t = str(t)
    if PREPROC["replace_url"]:  t = URL_RE.sub("URL", t)
    if PREPROC["replace_user"]: t = USER_RE.sub("USER", t)
    return re.sub(r"\s+", " ", t).strip()

def load_split(dataset: str, split: str) -> pd.DataFrame:
    path = f"{DATA_DIR}/{dataset}/{split}.csv"
    df = pd.read_csv(path)
    missing = {TEXT_COL, LABEL_COL} - set(df.columns)
    assert not missing, f"{path} is missing columns {missing}"
    df = df[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "text", LABEL_COL: "label"})
    raw = df["label"].astype(str).str.strip().str.lower()
    df["label"] = raw.map(LABEL_ALIASES)
    bad = sorted(raw[df["label"].isna()].unique())
    assert not bad, f"{path}: unknown labels {bad} - fix in the data audit / LABEL_ALIASES"
    labels = DATASETS[dataset]["labels"]
    extra = set(df["label"]) - set(labels)
    assert not extra, f"{path}: labels {extra} not expected for {dataset}"
    df["y"] = df["label"].map({l: i for i, l in enumerate(labels)})
    df["text"] = df["text"].map(preprocess)
    return df.reset_index(drop=True)

def file_md5(path):
    return hashlib.md5(open(path, "rb").read()).hexdigest()

# Sanity check: counts per split and label (copy into the report / compare with Member 1's audit)
for ds in DATASETS:
    for sp in ["train", "test"]:
        p = f"{DATA_DIR}/{ds}/{sp}.csv"
        if os.path.exists(p):
            d = load_split(ds, sp)
            print(f"{ds:10s} {sp:5s} n={len(d):6d}", d["label"].value_counts().to_dict())
        else:
            print(f"{ds:10s} {sp:5s} MISSING -> {p}")

arsarcasm  train n=  8437 {'neutral': 4262, 'negative': 2813, 'positive': 1362}
arsarcasm  test  n=  2110 {'neutral': 1078, 'negative': 716, 'positive': 316}
astd       train n= 27394 {'negative': 11241, 'neutral': 9483, 'positive': 6670}
astd       test  n=   661 {'negative': 336, 'neutral': 166, 'positive': 159}
ajgt       train n=  1440 {'negative': 720, 'positive': 720}
ajgt       test  n=   360 {'positive': 180, 'negative': 180}


## 4. Metrics (+ unit test)
* **F1-PN** = mean of positive-class F1 and negative-class F1, computed on the **full 3-class test set**
  (neutral examples kept, neutral F1 not averaged in).
* Every epoch also logs accuracy, macro-F1 and per-class precision / recall / F1 / support.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def compute_metrics(y_true, y_pred, labels):
    ids = list(range(len(labels)))
    m = {"accuracy": 100 * accuracy_score(y_true, y_pred),
         "macro_f1": 100 * f1_score(y_true, y_pred, labels=ids, average="macro", zero_division=0)}
    if "positive" in labels and "negative" in labels:
        pn = [labels.index("positive"), labels.index("negative")]
        m["f1_pn"] = 100 * f1_score(y_true, y_pred, labels=pn, average="macro", zero_division=0)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=ids, zero_division=0)
    m["per_class"] = {l: {"precision": 100 * p[i], "recall": 100 * r[i], "f1": 100 * f[i], "support": int(s[i])}
                      for i, l in enumerate(labels)}
    m["confusion_matrix"] = confusion_matrix(y_true, y_pred, labels=ids).tolist()
    return m

# Unit test (hand-computed): pos F1 = .5, neg F1 = 1, neu F1 = .5  ->  F1-PN = 75, macro-F1 = 66.67, acc = 66.67
_L = ["negative", "neutral", "positive"]; N, U, P = 0, 1, 2
_m = compute_metrics([P, P, N, N, U, U], [P, U, N, N, U, P], _L)
assert abs(_m["f1_pn"] - 75.0) < 1e-6 and abs(_m["macro_f1"] - 200/3) < 1e-6 and abs(_m["accuracy"] - 400/6) < 1e-6
print("metric unit test passed")

metric unit test passed


## 5. Seeding and model revisions
Model commit hashes are resolved once and stored, so every run uses the identical checkpoint.

In [ ]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
import transformers

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

REV_FILE = f"{PROJECT_DIR}/model_revisions.json"
def get_revisions():
    revs = json.load(open(REV_FILE)) if os.path.exists(REV_FILE) else {}
    changed = False
    for k, mid in MODELS.items():
        if k not in revs:
            if os.path.isdir(mid):
                revs[k] = None                       # local folder (testing)
            else:
                from huggingface_hub import HfApi
                revs[k] = HfApi().model_info(mid).sha
            changed = True
    if changed: json.dump(revs, open(REV_FILE, "w"), indent=2)
    return revs

REVISIONS = get_revisions()
print(json.dumps(REVISIONS, indent=2))

device: cuda
{
  "marbert": "88e1fa192dd723cf0b3563500aec46209762eb22",
  "arabert": "01f6b7d2d17afcd0f79498f26f47dfc6207b56ea"
}


## 6. One training run (resumable)
One run = one dataset × one model × one seed. After every epoch it writes, to Drive:
`metrics_per_epoch.jsonl`, `preds/epoch_XX_test_preds.csv` (gold, prediction, class probabilities),
and a resume checkpoint. A `DONE` file marks a finished run.

In [ ]:
import shutil
from torch.utils.data import DataLoader

class EncDataset(torch.utils.data.Dataset):
    def __init__(self, enc, y): self.enc, self.y = enc, y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}; item["labels"] = int(self.y[i]); return item

def _autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(USE_AMP and DEVICE == "cuda"))

@torch.no_grad()
def predict(model, loader):
    model.eval(); probs = []
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items() if k != "labels"}
        with _autocast():
            logits = model(**b).logits
        probs.append(torch.softmax(logits.float(), -1).cpu())
    return torch.cat(probs).numpy()

def train_one_run(dataset, model_key, seed, out_root=None, ckpt_root=None, max_train=None, epochs=None):
    out_root, ckpt_root = out_root or OUT_DIR, ckpt_root or CKPT_DIR
    epochs = epochs or HP["epochs"]
    labels = DATASETS[dataset]["labels"]
    run_dir  = f"{out_root}/{dataset}/{model_key}/seed{seed}"
    ckpt_path = f"{ckpt_root}/{dataset}/{model_key}/seed{seed}/last.pt"
    if os.path.exists(f"{run_dir}/DONE"):
        print(f"[skip] {dataset}/{model_key}/seed{seed} already done"); return
    os.makedirs(f"{run_dir}/preds", exist_ok=True); os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)

    set_seed(seed)
    train_df, test_df = load_split(dataset, "train"), load_split(dataset, "test")
    if max_train: train_df = train_df.sample(n=min(max_train, len(train_df)), random_state=seed).reset_index(drop=True)

    mid, rev = MODELS[model_key], REVISIONS[model_key]
    tok = AutoTokenizer.from_pretrained(mid, revision=rev)
    model = AutoModelForSequenceClassification.from_pretrained(mid, revision=rev, num_labels=len(labels)).to(DEVICE)
    enc = lambda df: tok(df["text"].tolist(), truncation=True, max_length=HP["max_len"])
    train_ds, test_ds = EncDataset(enc(train_df), train_df["y"].values), EncDataset(enc(test_df), test_df["y"].values)
    collate = DataCollatorWithPadding(tok)   # dynamic padding: pad to longest tweet in batch (logged)
    test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, collate_fn=collate)

    opt = torch.optim.Adam(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    start_epoch = 1
    metrics_file = f"{run_dir}/metrics_per_epoch.jsonl"

    if os.path.exists(ckpt_path):                                   # ---- resume
        ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["rng_cpu"])
        if DEVICE == "cuda" and ck.get("rng_cuda") is not None: torch.cuda.set_rng_state_all(ck["rng_cuda"])
        start_epoch = ck["epoch"] + 1
        if os.path.exists(metrics_file):                            # drop epochs logged after the checkpoint
            keep = [l for l in open(metrics_file) if json.loads(l)["epoch"] <= ck["epoch"]]
            open(metrics_file, "w").writelines(keep)
        print(f"[resume] {dataset}/{model_key}/seed{seed} from epoch {start_epoch}")
    else:
        if os.path.exists(metrics_file): os.remove(metrics_file)

    info = dict(protocol="historical: full training set, best epoch selected on TEST (as in [1])",
                dataset=dataset, model_key=model_key, model_id=mid, model_revision=rev, seed=seed,
                labels=labels, n_train=len(train_df), n_test=len(test_df),
                train_md5=file_md5(f"{DATA_DIR}/{dataset}/train.csv"), test_md5=file_md5(f"{DATA_DIR}/{dataset}/test.csv"),
                hp=HP, epochs=epochs, use_amp=USE_AMP, preprocessing=PREPROC, dynamic_padding=True, max_train=max_train,
                gpu=gpu_info, torch=torch.__version__, transformers=transformers.__version__,
                started=time.strftime("%Y-%m-%d %H:%M:%S"))
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)

    for epoch in range(start_epoch, epochs + 1):
        t0 = time.time(); model.train()
        g = torch.Generator(); g.manual_seed(seed * 1000 + epoch)   # data order depends only on (seed, epoch) -> safe resume
        loader = DataLoader(train_ds, batch_size=HP["batch_size"], shuffle=True, generator=g, collate_fn=collate)
        tot, n = 0.0, 0
        for b in loader:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            opt.zero_grad(set_to_none=True)
            with _autocast():
                loss = model(**b).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * b["labels"].size(0); n += b["labels"].size(0)

        probs = predict(model, test_loader); pred = probs.argmax(1)
        m = compute_metrics(test_df["y"].values, pred, labels)
        m.update(epoch=epoch, train_loss=tot / n, seconds=round(time.time() - t0, 1))
        with open(metrics_file, "a") as f: f.write(json.dumps(m) + "\n")
        pdf = pd.DataFrame({"idx": np.arange(len(test_df)), "gold": test_df["y"].values, "pred": pred})
        for i, l in enumerate(labels): pdf[f"p_{l}"] = probs[:, i].round(6)
        pdf.to_csv(f"{run_dir}/preds/epoch_{epoch:02d}_test_preds.csv", index=False)
        key = DATASETS[dataset]["metric"]
        print(f"{dataset}/{model_key}/seed{seed} ep{epoch:02d} loss={tot/n:.4f} {key}={m[key]:.2f} "
              f"macroF1={m['macro_f1']:.2f} ({m['seconds']}s)")

        ce = CKPT_EVERY.get(dataset, 1) if isinstance(CKPT_EVERY, dict) else CKPT_EVERY
        if ce and epoch % ce == 0 and epoch < epochs:                # ---- checkpoint (atomic write)
            tmp = ckpt_path + ".tmp"
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "rng_cpu": torch.get_rng_state(),
                        "rng_cuda": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None}, tmp)
            os.replace(tmp, ckpt_path)

    info["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2, ensure_ascii=False)
    open(f"{run_dir}/DONE", "w").write(info["finished"])
    shutil.rmtree(os.path.dirname(ckpt_path), ignore_errors=True)
    del model, opt; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 7. Smoke test
1 epoch on 256 AJGT tweets per model: checks the full pipeline and prints time per epoch. Written to a separate `outputs/smoke` folder.

In [ ]:
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    for mk in MODELS:
        shutil.rmtree(f"{PROJECT_DIR}/outputs/smoke/ajgt/{mk}", ignore_errors=True)
        train_one_run("ajgt", mk, seed=0, out_root=f"{PROJECT_DIR}/outputs/smoke",
                      ckpt_root=f"{PROJECT_DIR}/checkpoints/smoke", max_train=256, epochs=1)

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (12 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv01
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ajgt/arabert/seed0 ep01 loss=0.7073 accuracy=56.11 macroF1=52.44 (1.2s)


## 8. All historical runs (2 models × 3 datasets × 3 seeds = 18)
Order is **seed-first**: all 6 dataset×model runs for seed 42, then 43, then 44, so a complete first-seed table exists early.
To split work across several Colab/Kaggle accounts, give each account a different `ONLY` list and copy the `outputs/historical` folders together at the end.
Safe to re-run at any time: finished runs are skipped, interrupted runs resume.
To run only part of the grid in one session, set `ONLY`, e.g. `ONLY = [("astd", "marbert", 42)]`.

In [ ]:
RUN_ALL = True
ONLY = None   # e.g. [("astd", "marbert", 42), ("astd", "arabert", 42)]

if RUN_ALL:
    grid = [(d, m, s) for s in SEEDS for d in RUN_ORDER for m in MODELS]   # seed-first: one full seed of results ASAP
    if ONLY: grid = [g for g in grid if g in ONLY]
    for d, m, s in grid:
        train_one_run(d, m, s)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

astd/marbert/seed42 ep01 loss=0.6248 f1_pn=78.41 macroF1=57.75 (93.9s)
astd/marbert/seed42 ep02 loss=0.4666 f1_pn=79.61 macroF1=62.13 (94.3s)
astd/marbert/seed42 ep03 loss=0.4024 f1_pn=79.69 macroF1=62.49 (94.9s)
astd/marbert/seed42 ep04 loss=0.3474 f1_pn=79.85 macroF1=66.32 (94.4s)
astd/marbert/seed42 ep05 loss=0.2975 f1_pn=79.69 macroF1=65.34 (94.5s)
astd/marbert/seed42 ep06 loss=0.2592 f1_pn=80.17 macroF1=68.59 (95.2s)
astd/marbert/seed42 ep07 loss=0.2180 f1_pn=80.61 macroF1=67.31 (94.7s)
astd/marbert/seed42 ep08 loss=0.1833 f1_pn=80.37 macroF1=67.46 (95.1s)
astd/marbert/seed42 ep09 loss=0.1550 f1_pn=80.35 macroF1=68.28 (94.8s)
astd/marbert/seed42 ep10 loss=0.1350 f1_pn=81.52 macroF1=69.24 (94.8s)
astd/marbert/seed42 ep11 loss=0.1128 f1_pn=79.92 macroF1=68.73 (96.4s)
astd/marbert/seed42 ep12 loss=0.1013 f1_pn=79.86 macroF1=67.24 (96.2s)


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:1072: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:156.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


astd/marbert/seed42 ep13 loss=0.0887 f1_pn=80.87 macroF1=67.90 (94.6s)
astd/marbert/seed42 ep14 loss=0.0785 f1_pn=79.10 macroF1=67.19 (94.9s)
astd/marbert/seed42 ep15 loss=0.0739 f1_pn=79.21 macroF1=67.28 (94.8s)
astd/marbert/seed42 ep16 loss=0.0671 f1_pn=79.48 macroF1=67.55 (96.3s)
astd/marbert/seed42 ep17 loss=0.0617 f1_pn=79.48 macroF1=68.71 (95.4s)
astd/marbert/seed42 ep18 loss=0.0595 f1_pn=79.08 macroF1=66.37 (94.7s)
astd/marbert/seed42 ep19 loss=0.0546 f1_pn=79.28 macroF1=66.04 (95.0s)
astd/marbert/seed42 ep20 loss=0.0493 f1_pn=80.94 macroF1=69.72 (94.7s)
astd/marbert/seed42 ep21 loss=0.0462 f1_pn=79.71 macroF1=67.22 (95.3s)
astd/marbert/seed42 ep22 loss=0.0471 f1_pn=78.89 macroF1=65.58 (94.8s)
astd/marbert/seed42 ep23 loss=0.0448 f1_pn=79.89 macroF1=68.76 (94.4s)
astd/marbert/seed42 ep24 loss=0.0413 f1_pn=80.24 macroF1=67.23 (94.7s)
astd/marbert/seed42 ep25 loss=0.0371 f1_pn=79.67 macroF1=67.83 (94.8s)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (46 outputs left out)

ajgt/marbert/seed44 ep01 loss=0.6658 accuracy=86.94 macroF1=86.94 (5.3s)
ajgt/marbert/seed44 ep02 loss=0.5120 accuracy=91.94 macroF1=91.94 (5.3s)
ajgt/marbert/seed44 ep03 loss=0.2598 accuracy=92.22 macroF1=92.21 (5.3s)
ajgt/marbert/seed44 ep04 loss=0.1381 accuracy=92.50 macroF1=92.49 (5.3s)
ajgt/marbert/seed44 ep05 loss=0.0921 accuracy=92.78 macroF1=92.77 (5.3s)
ajgt/marbert/seed44 ep06 loss=0.0696 accuracy=93.06 macroF1=93.06 (5.2s)
ajgt/marbert/seed44 ep07 loss=0.0571 accuracy=94.17 macroF1=94.17 (5.3s)
ajgt/marbert/seed44 ep08 loss=0.0458 accuracy=93.61 macroF1=93.61 (5.3s)
ajgt/marbert/seed44 ep09 loss=0.0378 accuracy=93.33 macroF1=93.33 (5.3s)
ajgt/marbert/seed44 ep10 loss=0.0333 accuracy=94.17 macroF1=94.17 (5.3s)
ajgt/marbert/seed44 ep11 loss=0.0289 accuracy=93.06 macroF1=93.06 (5.3s)
ajgt/marbert/seed44 ep12 loss=0.0253 accuracy=93.61 macroF1=93.61 (5.3s)
ajgt/marbert/seed44 ep13 loss=0.0235 accuracy=93.89 macroF1=93.89 (5.3s)
ajgt/marbert/seed44 ep14 loss=0.0204 accuracy=92.78

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv01
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ajgt/arabert/seed44 ep01 loss=0.6401 accuracy=82.22 macroF1=82.22 (5.0s)
ajgt/arabert/seed44 ep02 loss=0.4637 accuracy=86.67 macroF1=86.65 (5.1s)
ajgt/arabert/seed44 ep03 loss=0.3332 accuracy=87.50 macroF1=87.50 (5.0s)
ajgt/arabert/seed44 ep04 loss=0.2618 accuracy=87.22 macroF1=87.17 (5.0s)
ajgt/arabert/seed44 ep05 loss=0.2052 accuracy=86.67 macroF1=86.65 (5.0s)
ajgt/arabert/seed44 ep06 loss=0.1744 accuracy=88.33 macroF1=88.33 (4.9s)
ajgt/arabert/seed44 ep07 loss=0.1373 accuracy=88.61 macroF1=88.61 (5.0s)
ajgt/arabert/seed44 ep08 loss=0.1215 accuracy=87.22 macroF1=87.22 (5.0s)
ajgt/arabert/seed44 ep09 loss=0.1002 accuracy=88.61 macroF1=88.61 (5.0s)
ajgt/arabert/seed44 ep10 loss=0.0824 accuracy=88.89 macroF1=88.89 (5.0s)
ajgt/arabert/seed44 ep11 loss=0.0767 accuracy=89.17 macroF1=89.17 (5.0s)
ajgt/arabert/seed44 ep12 loss=0.0564 accuracy=89.17 macroF1=89.17 (5.0s)
ajgt/arabert/seed44 ep13 loss=0.0539 accuracy=89.44 macroF1=89.44 (5.0s)
ajgt/arabert/seed44 ep14 loss=0.0452 accuracy=89.17

## 9. Published vs ours: reproduction verdict
For each run the **best epoch on test** (historical metric; earliest epoch on ties) is taken.
A target is **reproduced** if |gap| ≤ 2 pp **or** the published score lies within our seed range,
**and** MARBERT > AraBERT (mean over seeds) on that dataset.

In [11]:
def collect_runs(out_root=OUT_DIR):
    rows = []
    for d in DATASETS:
        key = DATASETS[d]["metric"]
        for mk in MODELS:
            for s in SEEDS:
                rd = f"{out_root}/{d}/{mk}/seed{s}"
                if not os.path.exists(f"{rd}/DONE"): continue
                ep = [json.loads(l) for l in open(f"{rd}/metrics_per_epoch.jsonl")]
                best = max(ep, key=lambda e: (e[key], -e["epoch"]))
                rows.append(dict(dataset=d, model=mk, seed=s, metric=key, best_epoch=best["epoch"],
                                 score=best[key], macro_f1_at_best=best["macro_f1"], accuracy_at_best=best["accuracy"]))
    return pd.DataFrame(rows)

runs = collect_runs()
if runs.empty:
    print("No finished runs yet.")
else:
    runs.to_csv(f"{OUT_DIR}/per_run_best_epoch.csv", index=False)
    display(runs) if IN_COLAB else print(runs.to_string())

    summ = []
    for d in DATASETS:
        means = {}
        for mk in MODELS:
            r = runs[(runs.dataset == d) & (runs.model == mk)]
            if r.empty: continue
            pub = DATASETS[d]["published"][mk]
            mean, sd = r.score.mean(), (r.score.std(ddof=1) if len(r) > 1 else float("nan"))
            means[mk] = mean
            summ.append(dict(dataset=d, metric=DATASETS[d]["metric"], model=mk, n_seeds=len(r),
                             published=pub, ours_mean=round(mean, 2), ours_sd=round(sd, 2),
                             ours_min=round(r.score.min(), 2), ours_max=round(r.score.max(), 2),
                             gap_pp=round(mean - pub, 2),
                             within_2pp=abs(mean - pub) <= 2, within_seed_range=r.score.min() <= pub <= r.score.max()))
        ranking = means.get("marbert", np.nan) > means.get("arabert", np.nan) if len(means) == 2 else None
        for row in summ:
            if row["dataset"] == d:
                row["marbert_gt_arabert"] = ranking
                row["reproduced"] = bool((row["within_2pp"] or row["within_seed_range"]) and ranking) if ranking is not None else None
    summary = pd.DataFrame(summ)
    summary.to_csv(f"{OUT_DIR}/reproduction_summary.csv", index=False)
    display(summary) if IN_COLAB else print(summary.to_string())
    incomplete = summary[summary.n_seeds < len(SEEDS)]
    if not incomplete.empty: print("NOTE: some targets have fewer than", len(SEEDS), "seeds - verdict is provisional.")

,dataset,model,seed,metric,best_epoch,score,macro_f1_at_best,accuracy_at_best
0,arsarcasm,marbert,42,f1_pn,5,72.113760,74.748703,77.109005
1,arsarcasm,marbert,43,f1_pn,11,73.176557,75.457292,77.535545
2,arsarcasm,marbert,44,f1_pn,10,71.973836,74.236525,76.303318
3,arsarcasm,arabert,42,f1_pn,12,67.630053,70.793725,73.554502
4,arsarcasm,arabert,43,f1_pn,10,68.216314,71.906440,75.260664
5,arsarcasm,arabert,44,f1_pn,13,67.740980,71.173040,74.360190
6,astd,marbert,42,f1_pn,10,81.515484,69.237273,73.071104
7,astd,marbert,43,f1_pn,6,80.483165,68.179253,72.465961
8,astd,marbert,44,f1_pn,5,80.688823,67.326384,72.314675
9,astd,arabert,42,f1_pn,17,75.797348,64.094784,68.229955


,dataset,metric,model,n_seeds,published,ours_mean,ours_sd,ours_min,ours_max,gap_pp,within_2pp,within_seed_range,marbert_gt_arabert,reproduced
0,arsarcasm,f1_pn,marbert,3,71.50,72.42,0.66,71.97,73.18,0.92,True,False,True,True
1,arsarcasm,f1_pn,arabert,3,63.50,67.86,0.31,67.63,68.22,4.36,False,False,True,False
2,astd,f1_pn,marbert,3,78.00,80.90,0.55,80.48,81.52,2.90,False,False,True,False
3,astd,f1_pn,arabert,3,72.00,75.83,0.17,75.68,76.01,3.83,False,False,True,False
4,ajgt,accuracy,marbert,3,96.11,94.35,0.32,94.17,94.72,-1.76,True,False,True,True
5,ajgt,accuracy,arabert,3,92.22,88.89,0.73,88.06,89.44,-3.33,False,False,True,False


## 10. Departures log
Creates `departures.md` in Drive the first time (edit it there; it is not overwritten).

In [12]:
DEP_FILE = f"{PROJECT_DIR}/departures.md"
if not os.path.exists(DEP_FILE):
    open(DEP_FILE, "w").write(f"""# Departures and assumptions - historical reproduction

| # | Paper [1] | Ours | Why / expected effect |
|---|---|---|---|
| 1 | AraBERT version not named | `{MODELS['arabert']}` @ `{REVISIONS['arabert']}` | Fixed 2020 base release; possible source of AraBERT gaps |
| 2 | MARBERT | `{MODELS['marbert']}` @ `{REVISIONS['marbert']}` | Original release, not MARBERTv2 |
| 3 | LR scheduler / warm-up unreported | Constant LR {HP['lr']}, no warm-up | Simplest reading of the paper |
| 4 | Precision not reported | fp16 mixed precision = {USE_AMP} | Speed on T4; small numeric differences |
| 5 | Padding not reported | Dynamic padding (max_len {HP['max_len']} truncation kept) | Speed; no effect on outputs with attention masks beyond float noise |
| 6 | Fine-tuning preprocessing not reported | mentions -> USER, URLs -> URL: {PREPROC} | Mirrors MARBERT pre-training normalisation (verify) |
| 7 | Seeds not reported | {SEEDS} | Three seeds per configuration |
| 8 | AJGT split not released | (fill in with Member 1) | Most likely cause of AJGT gaps |
| 9 | Hardware not reported | {gpu_info} | GPU non-determinism; small run-to-run noise |
| 10 | Classification head | HF `AutoModelForSequenceClassification` default (dropout 0.1, [CLS] pooler) | Standard BERT fine-tuning |
""")
    print("Created", DEP_FILE)
else:
    print(DEP_FILE, "exists - edit it in Drive")

Created /content/drive/MyDrive/marbert_repro/departures.md


In [13]:
from google.colab import drive; drive.mount("/content/drive")
import json, os, pandas as pd
R="/content/drive/MyDrive/marbert_repro/outputs/historical"
rows=[]
for d,key in [("arsarcasm","f1_pn"),("astd","f1_pn"),("ajgt","accuracy")]:
  for m in ["marbert","arabert"]:
    for s in [42,43,44]:
      f=f"{R}/{d}/{m}/seed{s}/metrics_per_epoch.jsonl"
      if os.path.exists(f):
        ep=[json.loads(l) for l in open(f)]; b=max(ep,key=lambda e:e[key])
        rows.append(dict(dataset=d,model=m,seed=s,epochs=len(ep),best_epoch=b["epoch"],score=round(b[key],2)))
print(pd.DataFrame(rows).to_string(index=False))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  dataset   model  seed  epochs  best_epoch  score
arsarcasm marbert    42      25           5  72.11
arsarcasm marbert    43      25          11  73.18
arsarcasm marbert    44      25          10  71.97
arsarcasm arabert    42      25          12  67.63
arsarcasm arabert    43      25          10  68.22
arsarcasm arabert    44      25          13  67.74
     astd marbert    42      25          10  81.52
     astd marbert    43      25           6  80.48
     astd marbert    44      25           5  80.69
     astd arabert    42      25          17  75.80
     astd arabert    43      25           7  75.68
     astd arabert    44      25           7  76.01
     ajgt marbert    42      25          22  94.72
     ajgt marbert    43      25          13  94.17
     ajgt marbert    44      25           7  94.17
     ajgt arabert    42      25           7  89.17
    